# tcrkit — repertoire metrics

Measures over a *set* of TCRs rather than one at a time.

| | answers | wraps |
|---|---|---|
| `diversity` | how evenly is the repertoire spread? (entropy, clonality) | scikit-bio |
| `morisita_horn` | how much do two repertoires overlap? | *computed here* |
| `generation_probability` | how likely was recombination to make this CDR3? | OLGA |
| `tcrdist` | how similar are two TCRs? | tcrdist3 |

These need an extra install, since OLGA and tcrdist3 are heavy:

```bash
pip install 'tcrkit[metrics]'      # or: uv sync --extra metrics
```

All four were checked against R's `vegan` and agree to ~1e-15.

In [1]:
import pandas as pd

import tcrkit

# four repertoires with deliberately different shapes
reps = pd.DataFrame(
    [{'sample': s, 'cdr3': c, 'count': n} for s, cs in {
        'expanded':   [('A', 480), ('B', 10), ('C', 5), ('D', 5)],   # one clone dominates
        'polyclonal': [('A', 125), ('B', 125), ('C', 125), ('D', 125)],
        'donor_t0':   [('A', 200), ('B', 100), ('E', 20)],
        'donor_t1':   [('A', 20),  ('B', 10),  ('E', 2)],   # same shape, 10x shallower
    }.items() for c, n in cs])
reps.head()

,sample,cdr3,count
0,expanded,A,480
1,expanded,B,10
2,expanded,C,5
3,expanded,D,5
4,polyclonal,A,125


---
## 1. `diversity` — entropy and clonality

One row per repertoire. `by=` splits them.

In [2]:
tcrkit.diversity(reps, by='sample', col_mapping={'clone': 'cdr3'}).round(3)

,repertoire,n_clones,n_cells,shannon,pielou_evenness,clonality,simpson,dominance,top_clone_fraction
0,donor_t0,3,320.0,1.198,0.756,0.244,0.508,0.492,0.625
1,donor_t1,3,32.0,1.198,0.756,0.244,0.508,0.492,0.625
2,expanded,4,500.0,0.302,0.151,0.849,0.078,0.922,0.960
3,polyclonal,4,500.0,2.000,1.000,0.000,0.750,0.250,0.250


- **`shannon`** is in **bits**. Four equally abundant clones carry exactly `log2(4) = 2`,
  which is what `polyclonal` scores.
- **`pielou_evenness`** divides that by `log2(n_clones)`, so it is 0–1 and comparable
  between samples of different richness. Raw entropy is not — it rises just from having
  more clones.
- **`clonality` = 1 − pielou_evenness.** 0 is perfectly even, 1 is a single clone.

You can skip the counting: pass raw clone labels (one row per cell) and it counts for you.

In [3]:
tcrkit.diversity(['A', 'A', 'A', 'B', 'C']).round(3)

,repertoire,n_clones,n_cells,shannon,pielou_evenness,clonality,simpson,dominance,top_clone_fraction
0,all,3,5.0,1.371,0.865,0.135,0.56,0.44,0.6


---
## 2. `morisita_horn` — overlap

0 when two repertoires share no clones, 1 when their clone-size distributions match.
`squeeze=True` for the single number.

In [4]:
print('identical:', tcrkit.morisita_horn({'A': 10, 'B': 5}, {'A': 10, 'B': 5}, squeeze=True))
print('disjoint :', tcrkit.morisita_horn({'A': 10, 'B': 5}, {'C': 10, 'D': 5}, squeeze=True))

tcrkit.morisita_horn(reps, by='sample', col_mapping={'clone': 'cdr3'}).round(3)

identical: 1.0
disjoint : 0.0


,donor_t0,donor_t1,expanded,polyclonal
donor_t0,1.000,1.000,0.857,0.632
donor_t1,1.000,1.000,0.857,0.632
expanded,0.857,0.857,1.000,0.427
polyclonal,0.632,0.632,0.427,1.000


`donor_t0` vs `donor_t1` is exactly **1.0** despite the 10× depth difference — they are
the same composition sampled harder. That depth-invariance is the reason to prefer this
over Jaccard. `long=True` gives one row per pair instead of the matrix.

---
## 3. `generation_probability` — OLGA

How likely V(D)J recombination was to produce a CDR3. A **low** pgen means it is hard to
make by chance, so seeing it in two people is far more surprising — the usual way to
weight how "public" a TCR is.

In [5]:
tcrkit.generation_probability(['CASSIRSSYEQYF', 'CASSYSIRGSRGEQYF', 'NOTACDR3'])[
    ['cdr3', 'pgen', 'log10_pgen', 'error']]

,cdr3,pgen,log10_pgen,error
0,CASSIRSSYEQYF,1.804634e-07,-6.743611,NaN
1,CASSYSIRGSRGEQYF,3.210328e-11,-10.493451,NaN
2,NOTACDR3,0.000000e+00,-inf,not an amino-acid sequence


Use `log10_pgen` for comparing and plotting — these span orders of magnitude. An
unreadable CDR3 is reported, not raised.

OLGA wants the **junction** (CDR3 *with* its C and F), which is what
`add_cdr3_junctions` produces, so the two compose. Passing V and/or J narrows the
recombinations it sums over — each one helps, independently:

In [6]:
pgen = lambda **kw: tcrkit.generation_probability('CASSIRSSYEQYF', **kw)['pgen'][0]
for label, kw in [('no genes', {}), ('V only', {'v': 'TRBV19'}),
                  ('J only', {'j': 'TRBJ2-7'}), ('both', {'v': 'TRBV19', 'j': 'TRBJ2-7'})]:
    print(f'{label:<9} {pgen(**kw):.3e}')

no genes  1.805e-07
V only    4.789e-08
J only    1.805e-07
both      4.789e-08


`chain='alpha'` and `species='mouse'` select a different OLGA model.

---
## 4. `tcrdist` — distances between TCRs

Small distance means similar CDR3 and similar germline genes, so plausibly the same
specificity. Expects `{chain}_cdr3` / `{chain}_v` / `{chain}_j` — the names
`build_tcr_pmhc` produces.

In [7]:
tcrs = pd.DataFrame({
    'beta_cdr3': ['CASSIRSSYEQYF', 'CASSIRSSYEQYF', 'CASSLGQAYEQYF', 'CASSYSIRGSRGEQYF'],
    'beta_v':    ['TRBV19', 'TRBV19', 'TRBV20-1', 'TRBV6-5'],
    'beta_j':    ['TRBJ2-7'] * 4,
}, index=['t1', 't1_copy', 't2', 't3'])

tcrkit.tcrdist(tcrs)

,t1,t1_copy,t2,t3
t1,0,0,99,128
t1_copy,0,0,99,128
t2,99,99,0,141
t3,128,128,141,0


`t1` and `t1_copy` are the same TCR, so 0. For anything but a small set `long=True` is
easier to filter and far smaller:

In [8]:
pairs = tcrkit.tcrdist(tcrs, long=True)
pairs[pairs.distance <= 100].sort_values('distance')

,i,j,distance
0,t1,t1_copy,0
3,t1_copy,t1,0
1,t1,t2,99
4,t1_copy,t2,99
6,t2,t1,99
7,t2,t1_copy,99


`chains=('alpha', 'beta')` uses both, summing over chains.

---
## Two naming traps

- **`clonality` means something else in immunarch.** Here it is 1 − Pielou evenness; in
  immunarch it is *clonal proportion* (the top-N share). `top_clone_fraction` is the
  column closest to that sense. Check which definition any figure you compare against
  used.
- **`simpson` is ambiguous in the literature.** Here (following scikit-bio) it is
  1 − Σp²; some papers use "Simpson" for Σp², which is the `dominance` column.

`morisita_horn` is the only metric computed inside tcrkit — no maintained Python package
provides it. It matches `vegan::vegdist(method = "horn")` to 4e-16.